This Jupyuter notebook includes the code for training a feed-forward neural network with different hyperparameters. It is divided in 2 stages:


Stage 1:
- Random search over architectures (1–3 layers, widths in {256,512,1024,2048})
- Random activations, dropout, batchnorm, lr, batch_size
- Max 50 epochs, early stopping + global pruning from epoch >= 30
- 100 trials
- Saves: arch_search_results_stage1.csv, arch_search_summary_stage1.json, best_model_stage1.pt

Stage 2:
- Loads top-K configs from Stage-1 CSV (by val_mse)
- Retrains each for up to 200 epochs with early stopping (no global pruning)
- K = 5
- Saves: arch_search_results_stage2.csv, arch_search_summary_stage2.json, best_model_stage2.pt

Both stages:
- Use log1p(Y_tx) as targets, standardized Xg_log1p as inputs
- Compute test MSE + Pearson for best-by-val model in that stage

In [ ]:
import os, json, math, random, csv, time, pathlib, ast
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import numpy as np
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split

Now, we set a global configuration. Here, we include a seed, important to ensure the reproducibility of the code, irrespective of who runs it or when. It also allows for a fair comparison between architectures, as we ensure that the weights are initialized the same way. We want to run the model and all the tensors in an NVIDIA CUDA-enabled GPU if available, which is essential to accelerate the analyses.

In [ ]:
SEED      = 42
TEST_FRAC = 0.15
VAL_FRAC  = 0.15
DEVICE    = "cuda" if torch.cuda.is_available() else "cpu"
AMP       = (DEVICE == "cuda")

This code is run in two stages. In stage 1, we train 100 neural networks with different combinations of hyperparameters for 50 epochs. The five top-best performing configurations, i.e., the ones with the lowest validation error (TOP_K_STAGE2) are selected for the second stage, where they are fine-tuned for 200 epochs.

We set a value for gradient clipping (maximum L2 norm) to keep training stability by avoiding exploding gradients. The model performance will be computed and logged every 5 epochs, which we found to be a compromise between keeping track of the performance often enough without slowing down too much the training. 

In [ ]:
STAGE        = 2          # set to 1 or 2
TOP_K_STAGE2 = 5          # number of best configs from Stage 1 to retrain in Stage 2

GRAD_CLIP    = 1.0
EVAL_EVERY   = 5          # evaluate every N epochs (both stages)

In [ ]:
# HPO search ranges (Stage 1)
BATCHES    = [128, 192, 256]
LRS        = [1e-3, 2e-3, 3e-3]
DROPOUTS   = [0.0, 0.1, 0.2]
BATCHNORMS = [False, True]
ACTS       = ["tanh", "relu", "gelu", "leakyrelu"]

# Architecture space: up to 3 layers, widths in this set
DEPTH_CHOICES = [1, 2, 3]
WIDTH_CHOICES = [256, 512, 1024, 2048]

# Stage-specific config
if STAGE == 1:
    N_TRIALS        = 100
    MAX_EPOCHS      = 50
    PATIENCE        = 5
    MIN_PRUNE_EPOCH = 30   # don't prune before this epoch
    PRUNE_FACTOR    = 1.5  # prune if val is > 1.5x global best
elif STAGE == 2:
    # N_TRIALS will be set after loading top-K from Stage-1 CSV
    N_TRIALS        = TOP_K_STAGE2
    MAX_EPOCHS      = 200
    PATIENCE        = 10
    MIN_PRUNE_EPOCH = None
    PRUNE_FACTOR    = None
else:
    raise ValueError("STAGE must be 1 or 2")